In [1]:
AUTO_INSTALL_PACKAGES = True
DATASET_URL = "https://www.kaggle.com/datasets/snehaanbhawal/resume-dataset"
DEFAULT_TOP_K = 5
SBERT_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

import importlib.util
import os
import re
import subprocess
import sys

def install_if_missing():
    packages = {
        "gradio": "gradio",
        "sentence_transformers": "sentence-transformers",
        "faiss": "faiss-cpu",
        "sklearn": "scikit-learn",
    }

    missing = [
        pip_name for module_name, pip_name in packages.items()
        if importlib.util.find_spec(module_name) is None
    ]

    if missing:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "--quiet", *missing]
        )

if AUTO_INSTALL_PACKAGES:
    install_if_missing()

import faiss
import gradio as gr
import numpy as np
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 83.2 MB/s eta 0:00:00


In [2]:
def get_sbert_model():
    if "model_sbert" in globals():
        return model_sbert
    if "model" in globals():
        return model
    return None


def find_resume_csv_path():
    preferred_paths = [
        "/kaggle/input/resume-dataset/Resume/Resume.csv",
        "/kaggle/input/datasets/snehaanbhawal/resume-dataset/Resume/Resume.csv",
    ]

    for path in preferred_paths:
        if os.path.exists(path):
            return path

    for root, _, files in os.walk("/kaggle/input"):
        if "Resume.csv" in files:
            return os.path.join(root, "Resume.csv")

    return None


def load_resume_dataset_fast():
    csv_path = find_resume_csv_path()

    if csv_path is None:
        raise FileNotFoundError(
            "Resume.csv not found. Add this Kaggle dataset first: " + DATASET_URL
        )

    df = pd.read_csv(csv_path)
    print("CSV loaded:", csv_path)
    print("Columns:", df.columns.tolist())

    text_col = "Resume_str" if "Resume_str" in df.columns else "Resume"
    label_col = "Category"

    df = df.dropna(subset=[text_col, label_col])

    resume_texts = df[text_col].astype(str).tolist()
    resume_labels = df[label_col].astype(str).tolist()

    return resume_texts, resume_labels


def build_runtime_if_needed():
    global model_sbert, index, resume_embeddings, resume_labels, resume_texts

    required = ["index", "resume_embeddings", "resume_labels", "resume_texts"]

    if get_sbert_model() is not None and all(name in globals() for name in required):
        return "Using existing AI/ML runtime objects."

    resume_texts, resume_labels = load_resume_dataset_fast()

    device_name = "cuda" if torch.cuda.is_available() else "cpu"
    print("Using device:", device_name)

    model_sbert = SentenceTransformer(SBERT_MODEL_NAME, device=device_name)

    resume_embeddings = model_sbert.encode(
        resume_texts,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True
    ).astype("float32")

    dimension = resume_embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(resume_embeddings)

    return f"Runtime built quickly from CSV: {len(resume_texts)} resumes, {len(set(resume_labels))} categories."


startup_status = build_runtime_if_needed()
startup_status


CSV loaded: /kaggle/input/datasets/snehaanbhawal/resume-dataset/Resume/Resume.csv
Columns: ['ID', 'Resume_str', 'Resume_html', 'Category']
Using device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/39 [00:00<?, ?it/s]

'Runtime built quickly from CSV: 2484 resumes, 24 categories.'

In [3]:
def clean_skill(skill):
    return re.sub(r"\s+", " ", skill.strip().lower())


def parse_required_skills(text):
    if not text or not text.strip():
        return []

    parts = re.split(r"[,;\n|]+", text)
    return sorted({clean_skill(p) for p in parts if clean_skill(p)})


def extract_skills_safe(text, top_n=20):
    common_stop_words = {
        "and", "are", "for", "from", "have", "with", "the", "this", "that",
        "will", "you", "your", "our", "job", "role", "work", "team", "using",
        "required", "experience", "candidate", "looking", "strong", "skills",
        "knowledge", "responsibilities", "ability", "years", "good"
    }

    words = re.findall(r"[a-zA-Z][a-zA-Z+#.\-]{2,}", text.lower())

    skills = [
        word for word in words
        if word not in common_stop_words
    ]

    return sorted(set(skills))[:top_n]


def resume_preview(text, max_chars=280):
    preview = re.sub(r"\s+", " ", str(text)).strip()
    return preview if len(preview) <= max_chars else preview[:max_chars - 3] + "..."


def predict_job_category(job_embedding):
    if "model_dl" not in globals() or "label_encoder" not in globals():
        return "Not available", "model_dl not available"

    try:
        current_device = globals().get(
            "device",
            torch.device("cuda" if torch.cuda.is_available() else "cpu")
        )

        model_dl.eval()
        job_tensor = torch.tensor(job_embedding, dtype=torch.float32).to(current_device)

        with torch.no_grad():
            probs = torch.softmax(model_dl(job_tensor), dim=1).cpu().numpy()[0]

        best_idx = int(np.argmax(probs))
        return str(label_encoder.classes_[best_idx]), f"{float(probs[best_idx]):.3f}"

    except Exception as e:
        return "Not available", str(e)


def calculate_attention_score(candidate_idx, job_embedding):
    if "attention_model" not in globals():
        return None

    try:
        current_device = globals().get(
            "device",
            torch.device("cuda" if torch.cuda.is_available() else "cpu")
        )

        attention_model.eval()

        resume_tensor = torch.tensor(
            resume_embeddings[candidate_idx].reshape(1, -1),
            dtype=torch.float32
        ).to(current_device)

        job_tensor = torch.tensor(job_embedding, dtype=torch.float32).to(current_device)

        with torch.no_grad():
            score = attention_model(resume_tensor, job_tensor).cpu().numpy()[0]

        return round(float(score), 4)

    except:
        return None


def calculate_hire_probability(similarity, skill_coverage, has_skills):
    if has_skills:
        probability = (0.65 * similarity) + (0.35 * skill_coverage)
    else:
        probability = 0.7 * similarity

    return round(float(np.clip(probability, 0, 1)), 3)


In [4]:
def find_candidates(job_description, required_skills_text, top_k):
    if not job_description or not job_description.strip():
        return pd.DataFrame(), "Please enter a job description.", ""

    sbert_model = get_sbert_model()
    job_embedding = sbert_model.encode([job_description], convert_to_numpy=True).astype("float32")

    top_k = min(int(top_k), len(resume_embeddings))
    distances, indices = index.search(job_embedding, top_k)

    required_skills = parse_required_skills(required_skills_text)

    if not required_skills:
        required_skills = extract_skills_safe(job_description, top_n=12)

    has_skills = len(required_skills) > 0

    predicted_category, category_confidence = predict_job_category(job_embedding)

    rows = []
    best_preview = ""

    for rank, idx in enumerate(indices[0], start=1):
        idx = int(idx)

        distance = float(distances[0][rank - 1])
        match_score = round(1 / (1 + max(distance, 0)), 3)

        semantic_similarity = float(
            cosine_similarity(
                resume_embeddings[idx].reshape(1, -1),
                job_embedding
            )[0][0]
        )

        candidate_skills = extract_skills_safe(resume_texts[idx], top_n=30)

        matched_skills = sorted(set(candidate_skills) & set(required_skills))
        missing_skills = sorted(set(required_skills) - set(candidate_skills))

        skill_coverage = len(matched_skills) / len(required_skills) if has_skills else 0

        hire_probability = calculate_hire_probability(
            semantic_similarity,
            skill_coverage,
            has_skills
        )

        attention_score = calculate_attention_score(idx, job_embedding)

        preview = resume_preview(resume_texts[idx])

        if rank == 1:
            best_preview = preview

        rows.append({
            "Rank": rank,
            "Candidate ID": idx,
            "Resume Category": resume_labels[idx],
            "Match Score": match_score,
            "Semantic Similarity": round(semantic_similarity, 3),
            "Skill Coverage": round(skill_coverage, 3),
            "Hire Probability": hire_probability,
            "Attention Score": attention_score if attention_score is not None else "N/A",
            "Matched Skills": ", ".join(matched_skills) if matched_skills else "N/A",
            "Missing Skills": ", ".join(missing_skills) if missing_skills else "N/A",
            "Resume Preview": preview
        })

    result_df = pd.DataFrame(rows)

    summary = f"""
Predicted job category: **{predicted_category}**  
Category confidence: **{category_confidence}**  
Required skills used: **{', '.join(required_skills) if required_skills else 'None'}**  
Candidates searched: **{len(resume_embeddings)}**  
Returned top candidates: **{len(result_df)}**
"""

    return result_df, summary, best_preview


def recruitment_chat(message, history):
    history = history or []
    msg = (message or "").strip()

    if not msg:
        return "", history

    lower = msg.lower()

    if "dataset" in lower:
        reply = f"The system uses the Kaggle Resume Dataset: {DATASET_URL}"
    elif "model" in lower or "algorithm" in lower:
        reply = "The system uses SBERT embeddings with FAISS search."
    elif "skill" in lower:
        reply = "Enter required skills as comma-separated text."
    elif "candidate" in lower or "rank" in lower:
        reply = "Use the Candidate Finder tab, paste a job description, select Top K, and click Find Candidates."
    elif "probability" in lower:
        reply = "Hire probability is based on semantic similarity and skill coverage."
    else:
        reply = "I can help with dataset, model, skills, candidate ranking, and hire probability."

    history.append({"role": "user", "content": msg})
    history.append({"role": "assistant", "content": reply})

    return "", history


def clear_chat():
    return []


In [5]:
custom_css = """
body{
    background:#020617;
}

/* =========================================================
ULTRA TRANSPARENT GLASS CONTAINER
========================================================= */

.block{

    background:rgba(15, 23, 42, 0.18) !important;

    backdrop-filter:blur(20px) !important;

    -webkit-backdrop-filter:blur(20px) !important;

    border:1px solid rgba(255,255,255,0.05) !important;

    border-radius:24px !important;

    box-shadow:
    0 8px 32px rgba(0,0,0,0.22) !important;
}

/* =========================================================
MAIN CONTAINER
========================================================= */

.gradio-container{
    max-width:1500px !important;
    margin:auto;
}

/* =========================================================
INPUT BOXES
========================================================= */

textarea,
input{
    border-radius:16px !important;
    border:1px solid #374151 !important;
    background:#081126 !important;
    color:white !important;

    box-shadow:
    inset 0 0 10px rgba(255,255,255,0.03),
    0 0 10px rgba(0,0,0,0.35);

    transition:0.3s;
}

textarea:focus,
input:focus{
    border:1px solid #c084fc !important;

    box-shadow:
    0 0 15px rgba(192,132,252,0.35) !important;
}

/* =========================================================
JOB DESCRIPTION LABEL
========================================================= */

#job_box.wrap label{

    background:linear-gradient(
    135deg,
    #11998e,
    #38ef7d
) !important;

    color:white !important;

    padding:10px 16px !important;

    border-radius:12px !important;

    width:fit-content !important;

    font-weight:700 !important;

    box-shadow:
    0 0 15px rgba(142,45,226,0.35);
}

/* =========================================================
REQUIRED SKILLS LABEL
========================================================= */

#skills_box .block-title{

    background:linear-gradient(
        135deg,
        #00c9ff,
        #92fe9d
    ) !important;

    color:#081126 !important;

    padding:10px 16px !important;

    border-radius:12px !important;

    width:fit-content !important;

    font-weight:700 !important;

    box-shadow:
    0 0 15px rgba(0,201,255,0.35);
}

/* =========================================================
TOP K LABEL
========================================================= */

#topk_box .block-title{

    background:linear-gradient(
        135deg,
        #f7971e,
        #ffd200
    ) !important;

    color:#111827 !important;

    padding:10px 16px !important;

    border-radius:12px !important;

    width:fit-content !important;

    font-weight:800 !important;

    box-shadow:
    0 0 15px rgba(255,210,0,0.35);
}

/* =========================================================
TOP K SLIDER
========================================================= */

#topk_box input[type="range"]{
    accent-color:#22c55e !important;
}
#topk_box input[type="range"]::-webkit-slider-runnable-track{
    background:linear-gradient(
        90deg,
        #22c55e,
        #14b8a6,
        #06b6d4
    ) !important;
    height:10px !important;
    border-radius:999px !important;
}

/* =========================================================
PREVIEW LABEL
========================================================= */

#preview_box.block-title{

    background:linear-gradient(
    135deg,
    #11998e,
    #38ef7d
) !important;

    color:white !important;

    padding:10px 16px !important;

    border-radius:12px !important;

    width:fit-content !important;

    font-weight:700 !important;

    box-shadow:
    0 0 15px rgba(142,45,226,0.35);
}

/* =========================================================
BUTTONS
========================================================= */

button{
    border-radius:14px !important;
    font-weight:bold !important;
    transition:0.3s;
    border:none !important;
    color:white !important;
}

button:hover{
    transform:scale(1.03);
}

/* =========================================================
FIND BUTTON
========================================================= */

button.primary{

    background:linear-gradient(
        135deg,
        #0f172a,
        #1e293b,
        #334155
    ) !important;

    color:#38bdf8 !important;

    border:1px solid #38bdf8 !important;

    box-shadow:
    0 0 18px rgba(56,189,248,0.25);
}

/* =========================================================
SEND BUTTON
========================================================= */

#send_btn{

    background:linear-gradient(
        135deg,
        #22c55e,
        #166534,
        #0a0f0d
    ) !important;

    box-shadow:
    0 0 15px rgba(34,197,94,0.30);
}

/* =========================================================
CLEAR BUTTON
========================================================= */

#clear_btn{

    background:linear-gradient(
        135deg,
        #ef4444,
        #991b1b,
        #0f0a0a
    ) !important;

    box-shadow:
    0 0 15px rgba(239,68,68,0.30);
}

/* =========================================================
DATAFRAME
========================================================= */

table{
    border-radius:18px !important;
    overflow:hidden !important;
}

/* =========================================================
CHATBOT
========================================================= */

.message-wrap{
    border-radius:16px !important;
}

/* =========================================================
TAB ACTIVE COLOR CHANGE
========================================================= */

.tab-nav button,
button[role="tab"]{
    color:#e5e7eb !important;
    background:transparent !important;
    font-weight:800 !important;
    border-radius:14px 14px 0 0 !important;
    border:none !important;
    box-shadow:none !important;
}

.tab-nav button:hover,
button[role="tab"]:hover{
    color:#facc15 !important;
    background:rgba(250,204,21,0.08) !important;
    transform:none !important;
}

/* =========================================================
FOOTER
========================================================= */

footer{
    display:none !important;
}
"""

with gr.Blocks(
    title="AI Assisted Smart Recruitment System",
    theme=gr.themes.Soft(
        primary_hue="blue",
        neutral_hue="slate"
    ),
    css=custom_css
) as app:

    # ==================================================
    # HERO SECTION
    # ==================================================

    gr.HTML("""

    <div style="
        text-align:center;
        padding:18px;
        border-radius:24px;
        margin-bottom:18px;

        background:
        linear-gradient(
            90deg,
            #c000c0 0%,
            #7c3aed 35%,
            #2563eb 70%,
            #0891b2 100%
        );

        position:relative;
        overflow:hidden;
    ">

        <div style="
            position:absolute;
            bottom:-25px;
            left:0;
            width:100%;
            height:70px;
            background:#020617;
            border-radius:100% 100% 0 0;
            opacity:0.35;
        ">
        </div>

        <h1 style="
            color:white;
            font-size:42px;
            font-weight:800;
            margin-bottom:6px;
            position:relative;
            z-index:2;
        ">
            AI Assisted Smart Recruitment System
        </h1>

        <p style="
            color:#e2e8f0;
            font-size:18px;
            position:relative;
            z-index:2;
        ">
            Smart Recruitment AI: Where Deep Learning Meets Hiring Intelligence
        </p>

    </div>
    """)

    # ==================================================
    # TAB 1
    # ==================================================

    with gr.Tab("🔍 Candidate Finder"):

        with gr.Row():

            with gr.Column(scale=2):

                job_input = gr.Textbox(
                    label="📄 Job Description",
                    lines=8,
                    placeholder="Enter role, responsibilities, skills...",
                    elem_id="job_box"
                )

                skills_input = gr.Textbox(
                    label="🧠 Required Skills",
                    lines=2,
                    placeholder="python, machine learning, sql, nlp",
                    elem_id="skills_box"
                )

                top_k_input = gr.Slider(
                    label="🎯 Top K Candidates",
                    minimum=1,
                    maximum=20,
                    value=DEFAULT_TOP_K,
                    step=1,
                    elem_id="topk_box"
                )

                find_button = gr.Button(
                    "🚀 Find Best Candidates",
                    variant="primary"
                )

            with gr.Column(scale=1):

                summary_output = gr.Markdown()

                preview_output = gr.Textbox(
                    label="🏆 Best Candidate Resume Preview",
                    lines=14,
                    interactive=False,
                    elem_id="preview_box"
                )

        results_output = gr.Dataframe(
            label="📊 Top K Candidate Ranking",
            interactive=False
        )

        find_button.click(
            fn=find_candidates,
            inputs=[job_input, skills_input, top_k_input],
            outputs=[results_output, summary_output, preview_output]
        )

    # ==================================================
    # TAB 2
    # ==================================================

    with gr.Tab("🤖 Recruitment Chatbot"):

        chatbot = gr.Chatbot(
            label="AI Assistant",
            type="messages",
            height=450
        )

        chat_message = gr.Textbox(
            label="💬 Message",
            placeholder="Ask about ranking, skills, AI models..."
        )

        with gr.Row():

            send_button = gr.Button(
                "📨 Send",
                elem_id="send_btn"
            )

            clear_button = gr.Button(
                "🗑 Clear",
                elem_id="clear_btn"
            )

        chat_message.submit(
            fn=recruitment_chat,
            inputs=[chat_message, chatbot],
            outputs=[chat_message, chatbot]
        )

        send_button.click(
            fn=recruitment_chat,
            inputs=[chat_message, chatbot],
            outputs=[chat_message, chatbot]
        )

        clear_button.click(
            fn=clear_chat,
            inputs=None,
            outputs=chatbot,
            queue=False
        )

    # ==================================================
    # CUSTOM FOOTER
    # ==================================================

    gr.HTML("""
    <div style="
        margin-top:30px;
        padding:18px 10px;
        text-align:center;
        color:#cbd5e1;
        font-size:13px;
        font-weight:580;
    ">
        <span>© 2026 AI Recruitment System</span>
        <span style="color:#64748b; margin:0 10px;">•</span>

        <span style="display:inline-flex; align-items:center; gap:6px;">
            <span>⚡</span>
            <span>Built with Gradio</span>
        </span>

        <span style="color:#64748b; margin:0 10px;">•</span>

        <span style="display:inline-flex; align-items:center; gap:6px;">
            <span>⚙️</span>
            <span>Settings</span>
        </span>
    </div>
    """)


app.launch(share=True)

/tmp/ipykernel_23/1166009476.py:301: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_23/1166009476.py:301: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_23/1166009476.py:436: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://06e4c0de74f8deabe5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
